# MongoDB: безопасное чтение документов


Показать обнаружение коллекций, выбор полей, ограниченный поиск, группировку и профилирование схемы.

Нужен пакет `pymongo` и переменная `SOURCE_MONGODB_URI`. Без неё каждая ячейка сообщает о пропуске; URI никогда не печатается.

Ограничения: Для осторожной разведки внешних документов и проверки формы перед загрузкой. Учитывайте неоднородность документов, индексы и лимит; не выполняйте неограниченный поиск.


## Основные понятия
Документы могут иметь разные поля, поэтому запрос ограничивает размер и явно задаёт проекцию. URI не выводится.


In [ ]:
import os
from pymongo import MongoClient

## Примеры


**1. Доступность**

Проверьте наличие URI без печати секрета.



In [ ]:
uri = os.getenv("SOURCE_MONGODB_URI")
print("MongoDB настроен:", bool(uri))

**2. Коллекции**

Получите ограниченный список коллекций базы.



In [ ]:
if not os.getenv("SOURCE_MONGODB_URI"):
    print("Пример пропущен: SOURCE_MONGODB_URI не задан")
else:
    with MongoClient(os.environ["SOURCE_MONGODB_URI"], serverSelectionTimeoutMS=5000) as client:
        print(sorted(client["dota_events"].list_collection_names())[:30])


**3. Поиск с проекцией**

Верните только нужные поля и не более десяти документов.



In [ ]:
if not os.getenv("SOURCE_MONGODB_URI"):
    print("Пример пропущен: SOURCE_MONGODB_URI не задан")
else:
    with MongoClient(os.environ["SOURCE_MONGODB_URI"], serverSelectionTimeoutMS=5000) as client:
        rows = list(client["dota_events"]["kill_events"].find({"time": {"$gte": 0}}, {"_id": 0, "match_id": 1, "time": 1}).sort("time", 1).limit(10))
        print(*rows, sep="\n")


**4. Группировка**

Посчитайте число документов по идентификатору матча.



In [ ]:
if not os.getenv("SOURCE_MONGODB_URI"):
    print("Пример пропущен: SOURCE_MONGODB_URI не задан")
else:
    pipeline = [{"$match": {"match_id": {"$exists": True}}}, {"$group": {"_id": "$match_id", "events": {"$sum": 1}}}, {"$sort": {"events": -1}}, {"$limit": 10}]
    with MongoClient(os.environ["SOURCE_MONGODB_URI"], serverSelectionTimeoutMS=5000) as client:
        print(*client["dota_events"]["kill_events"].aggregate(pipeline), sep="\n")


**5. Профиль схемы**

Посчитайте наличие ключевых полей на ограниченной выборке.



In [ ]:
if not os.getenv("SOURCE_MONGODB_URI"):
    print("Пример пропущен: SOURCE_MONGODB_URI не задан")
else:
    with MongoClient(os.environ["SOURCE_MONGODB_URI"], serverSelectionTimeoutMS=5000) as client:
        rows = list(client["dota_events"]["kill_events"].find({}, {"_id": 0, "match_id": 1, "time": 1}).limit(100))
        print({"rows": len(rows), "with_match_id": sum("match_id" in row for row in rows), "with_time": sum("time" in row for row in rows)})


**6. Параметр диапазона**

Передайте границу времени как значение фильтра, а не собирайте строку запроса.



In [ ]:
if not os.getenv("SOURCE_MONGODB_URI"):
    print("Пример пропущен: SOURCE_MONGODB_URI не задан")
else:
    lower_time = 0
    with MongoClient(os.environ["SOURCE_MONGODB_URI"], serverSelectionTimeoutMS=5000) as client:
        rows = list(client["dota_events"]["kill_events"].find({"time": {"$gte": lower_time}}, {"_id": 0, "time": 1}).limit(10))
        print(*rows, sep="\n")
